# Retail Sales Analysis — KPI & Profitability Study

**Dataset:** Retail Sales (5,000 rows · 257 Products · 3 Categories · 2 States · 2014–2015)  
**Goal:** Understand what drives revenue and profit — and where the business is bleeding money.


## 1. Problem Statement

A retail business operating across NSW and VIC (Australia) sells products across three categories — **Office Supplies**, **Technology**, and **Furniture** — to four customer segments.

The raw dataset contained pre-calculated financial columns that were inconsistent with the raw pricing data. This analysis re-derives all financial KPIs from source columns (Cost Price, Retail Price, Quantity, Discount %) to ensure accuracy.

**Key questions this notebook answers:**
- Are any products actively destroying margin?
- Is revenue growth aligned with profit growth?
- Which categories and products are the real profit drivers?
- Is discounting hurting the bottom line?


## 2. Data Loading

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.style.use('ggplot')

sales = pd.read_csv('retail_sales_dataset/data.csv')
print(f"Rows: {sales.shape[0]:,} | Columns: {sales.shape[1]}")


## 3. Data Understanding

Quick structural scan before any transformation.


In [ ]:
# Column names, types, and null counts at a glance
sales.info()


In [ ]:
# Duplicate Order IDs are expected — each row is a line item within an order
print(f"Total rows       : {len(sales):,}")
print(f"Unique orders    : {sales['Order No'].nunique():,}")
print(f"Unique products  : {sales['Product Name'].nunique():,}")
print(f"Categories       : {sales['Product Category'].unique().tolist()}")
print(f"Customer types   : {sales['Customer Type'].unique().tolist()}")
print(f"States           : {sales['State'].unique().tolist()}")


**Notes on source columns:**
- `Profit Margin`, `Sub Total`, `Discount $`, `Order Total`, `Total` — pre-calculated but inconsistent with raw pricing. Dropped and recomputed.
- `Customer Name`, `Address`, `Account Manager` — PII / not useful for analysis. Dropped.
- `Ship Mode`, `Product Container`, `Order Priority` — operational metadata, not relevant to financial KPIs. Dropped.


## 4. Data Cleaning

In [ ]:
# Drop columns with pre-calculated errors and irrelevant metadata
cols_to_drop = [
    'Profit Margin', 'Sub Total', 'Discount $', 'Order Total', 'Total',  # erroneous pre-calcs
    'Customer Name', 'Address', 'Account Manager',                         # PII
    'Ship Mode', 'Product Container', 'Order Priority'                     # operational, out of scope
]
sales.drop(columns=cols_to_drop, inplace=True)


In [ ]:
# Fix types: strip currency/percent symbols and parse dates
sales['Retail Price'] = sales['Retail Price'].str.replace('$', '', regex=False).astype('float64')
sales['Cost Price']   = sales['Cost Price'].str.replace('$', '', regex=False).astype('float64')
sales['Discount %']   = sales['Discount %'].str.replace('%', '', regex=False).astype('float64')
sales['Order Date']   = pd.to_datetime(sales['Order Date'], dayfirst=True)


In [ ]:
# Derive all financial KPIs from source columns
#
# Formula chain:
#   Gross Revenue  = Retail Price × Quantity
#   Discount Amt   = Gross Revenue × (Discount % / 100)
#   Net Revenue    = Gross Revenue − Discount Amt
#   COGS           = Cost Price × Quantity
#   Gross Profit   = Net Revenue − COGS
#   Gross Margin % = (Gross Profit / Net Revenue) × 100
#   Unit Profit    = Gross Profit / Quantity  ← used in discount impact analysis

sales['Gross Revenue']  = sales['Retail Price'] * sales['Order Quantity']
sales['Discount Amt']   = sales['Gross Revenue'] * (sales['Discount %'] / 100)
sales['Net Revenue']    = sales['Gross Revenue'] - sales['Discount Amt']
sales['COGS']           = sales['Cost Price'] * sales['Order Quantity']
sales['Gross Profit']   = sales['Net Revenue'] - sales['COGS']
sales['Gross Margin %'] = (sales['Gross Profit'] / sales['Net Revenue']) * 100
sales['Unit Profit']    = sales['Gross Profit'] / sales['Order Quantity']

print("Derived columns added. Clean dataset shape:", sales.shape)


## 5. Exploratory Data Analysis

In [ ]:
# Financial summary of the cleaned dataset
sales[['Gross Revenue', 'Net Revenue', 'COGS', 'Gross Profit', 'Gross Margin %']].describe().round(2)


In [ ]:
# Revenue split by customer segment
sales.groupby('Customer Type')['Net Revenue'].sum().round(2).sort_values(ascending=False)


In [ ]:
# Revenue split by state
sales.groupby('State')['Net Revenue'].sum().round(2).sort_values(ascending=False)


## 6. KPI Analysis

### KPI 1 — What proportion of products operate at negative margins, and how much do they impact overall profitability?


In [ ]:
# Weighted margin per product (sum COGS and revenue first, then compute %)
product_totals = sales.groupby('Product Name').agg(
    Net_Revenue=('Net Revenue', 'sum'),
    COGS=('COGS', 'sum')
)
product_totals['Margin %'] = ((product_totals['Net_Revenue'] - product_totals['COGS']) / product_totals['Net_Revenue']) * 100

negative_margin_products = product_totals[product_totals['Margin %'] < 0]
total_products = product_totals.shape[0]

print(f"Products with negative margin : {len(negative_margin_products)} / {total_products} ({len(negative_margin_products)/total_products*100:.2f}%)")
print()
print(negative_margin_products.sort_values('Margin %').round(2))


In [ ]:
# Quantify the financial drag from loss-making line items
negative_rows   = sales[sales['Gross Profit'] < 0]
total_loss      = negative_rows['Gross Profit'].sum()
overall_profit  = sales['Gross Profit'].sum()
drag_pct        = (abs(total_loss) / overall_profit) * 100

print(f"Total loss from negative-margin line items : ${abs(total_loss):,.2f}")
print(f"Overall company gross profit               : ${overall_profit:,.2f}")
print(f"Profitability drag                         : {drag_pct:.3f}%")


### KPI 2 — How are revenue and profit trending over time, and are they aligned?


In [ ]:
monthly_revenue = sales.resample('ME', on='Order Date')['Net Revenue'].sum()
monthly_profit  = sales.resample('ME', on='Order Date')['Gross Profit'].sum()
monthly_trend   = pd.concat([monthly_revenue, monthly_profit], axis=1)
monthly_trend.columns = ['Net Revenue', 'Gross Profit']
monthly_trend.round(2)


### KPI 3 — Which categories generate high revenue but low or negative profit?


In [ ]:
cat_rev    = sales.groupby('Product Category')['Net Revenue'].sum().round(2)
cat_profit = sales.groupby('Product Category')['Gross Profit'].sum().round(2)
cat_summary = pd.concat([cat_rev, cat_profit], axis=1)
cat_summary.columns = ['Net Revenue', 'Gross Profit']
cat_summary['Profit Margin %'] = (cat_summary['Gross Profit'] / cat_summary['Net Revenue'] * 100).round(2)
cat_summary.sort_values('Net Revenue', ascending=False)


### KPI 4 — Are top-selling products contributing positively to profit?


In [ ]:
# Build a product-level summary
products_df = sales.groupby('Product Name').agg(
    Total_Revenue=('Net Revenue', 'sum'),
    Total_Qty=('Order Quantity', 'sum'),
    Order_Count=('Order No', 'count'),
    Total_Profit=('Gross Profit', 'sum')
).reset_index()

products_df['Profit Margin %'] = (products_df['Total_Profit'] / products_df['Total_Revenue'] * 100).round(2)
products_df['Avg Profit / Unit'] = (products_df['Total_Profit'] / products_df['Total_Qty']).round(2)

# Top 10 by revenue and their profit margin
top10_revenue = products_df.sort_values('Total_Revenue', ascending=False).head(10)
top10_revenue[['Product Name', 'Total_Revenue', 'Total_Profit', 'Profit Margin %']].reset_index(drop=True)


### KPI 5 — Which products consistently generate losses, and what's driving it?


In [ ]:
# Bottom 10 products by total profit
bottom10_profit = products_df.sort_values('Total_Profit').head(10)
bottom10_profit[['Product Name', 'Total_Revenue', 'Total_Qty', 'Total_Profit', 'Profit Margin %']].reset_index(drop=True)


In [ ]:
# Are losses driven by high discounts?
loss_product_names = products_df[products_df['Total_Profit'] < 0]['Product Name'].tolist()
loss_orders = sales[sales['Product Name'].isin(loss_product_names)]

print(f"Avg discount on loss-making products : {loss_orders['Discount %'].mean():.2f}%")
print(f"Avg discount across all products     : {sales['Discount %'].mean():.2f}%")


### KPI 6 — How does discounting impact profitability — is there a break-even threshold?


In [ ]:
# Average unit profit by discount band
sales['Discount Band'] = pd.cut(
    sales['Discount %'],
    bins=[-1, 0, 5, 10, 15, 20, 100],
    labels=['0%', '1-5%', '6-10%', '11-15%', '16-20%', '>20%']
)

discount_impact = sales.groupby('Discount Band', observed=True).agg(
    Avg_Unit_Profit=('Unit Profit', 'mean'),
    Order_Count=('Order No', 'count')
).round(2)

print(discount_impact)


### KPI 7 — Is business growth driven by sustainable profit, or just increased revenue?


In [ ]:
# Compute rolling profit margin % over time
monthly_trend['Profit Margin %'] = (monthly_trend['Gross Profit'] / monthly_trend['Net Revenue'] * 100).round(2)
monthly_trend[['Net Revenue', 'Gross Profit', 'Profit Margin %']].round(2)


## 7. Visualizations

Each chart is tied to a specific KPI and answers a specific question.


**Chart 1 — Monthly Revenue & Profit Trend** *(KPIs 2 & 7)*  
*Why it exists: Checks whether profit scales with revenue or decouples — the earliest signal of a margin problem.*

In [ ]:
fig, ax1 = plt.subplots(figsize=(16, 8))

ax1.plot(monthly_trend.index, monthly_trend['Net Revenue'],  color='steelblue', linewidth=2, label='Net Revenue')
ax1.plot(monthly_trend.index, monthly_trend['Gross Profit'], color='seagreen',  linewidth=2, linestyle='--', label='Gross Profit')
ax1.set_ylabel('Amount ($)')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
ax1.set_xlabel('Month')

ax2 = ax1.twinx()
ax2.bar(monthly_trend.index, monthly_trend['Profit Margin %'], width=20, alpha=0.25, color='goldenrod', label='Profit Margin %')
ax2.set_ylabel('Profit Margin %')
ax2.set_ylim(0, 100)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.title('Monthly Revenue, Profit & Margin %')
# Set x-ticks on the primary axis (ax1)
ax1.set_xticks(monthly_trend.index)
ax1.set_xticklabels(
    monthly_trend.index.strftime('%b %Y'), 
    rotation=45, 
    ha='right', 
    fontsize=8
)
plt.tight_layout()
plt.show()


**Chart 2 — Revenue vs. Profit by Category** *(KPI 3)*  
*Why it exists: Surfaces categories that look strong on revenue but are thin or negative on profit.*

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

cat_summary_sorted = cat_summary.sort_values('Net Revenue', ascending=False)
x = range(len(cat_summary_sorted))
width = 0.35

bars_rev  = ax.bar([i - width/2 for i in x], cat_summary_sorted['Net Revenue'],  width, label='Net Revenue',  color='steelblue')
bars_prof = ax.bar([i + width/2 for i in x], cat_summary_sorted['Gross Profit'], width, label='Gross Profit', color='seagreen')

# Annotate profit margin % above each profit bar
for bar, margin in zip(bars_prof, cat_summary_sorted['Profit Margin %']):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() * 1.02,
            f'{margin:.1f}%',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_xticks(list(x))
ax.set_xticklabels(cat_summary_sorted.index, rotation=0)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v/1e6:.1f}M'))
ax.set_ylabel('Amount ($)')
ax.set_title('Net Revenue vs. Gross Profit by Product Category')
ax.legend()
plt.tight_layout()
plt.show()


**Chart 3 — Top 10 Products by Revenue with Profit Margin** *(KPI 4)*  
*Why it exists: Identifies whether the highest-revenue products actually deliver profit.*

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 5))

top10_sorted = top10_revenue.sort_values('Total_Revenue', ascending=True)

bars = ax1.barh(top10_sorted['Product Name'], top10_sorted['Total_Revenue'], color='steelblue', label='Total Revenue')
ax1.set_xlabel('Total Net Revenue ($)')
ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v/1e3:.0f}K'))

# Overlay profit margin % as text
for bar, margin in zip(bars, top10_sorted['Profit Margin %']):
    color = 'seagreen' if margin >= 0 else 'crimson'
    ax1.text(bar.get_width() * 1.01, bar.get_y() + bar.get_height() / 2,
             f'{margin:.1f}%', va='center', fontsize=9, color=color, fontweight='bold')

ax1.set_title('Top 10 Products by Revenue — Profit Margin % Annotated')
plt.tight_layout()
plt.show()


**Chart 4 — Discount % vs. Unit Profit** *(KPI 6)*  
*Why it exists: Determines whether there is a discount threshold beyond which profitability collapses.*

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Scatter: raw relationship
ax1.scatter(sales['Discount %'], sales['Unit Profit'], alpha=0.3, color='steelblue', s=15)
ax1.axhline(0, color='crimson', linewidth=1.2, linestyle='--', label='Break-even')
ax1.set_xlabel('Discount %')
ax1.set_ylabel('Unit Profit ($)')
ax1.set_title('Discount % vs. Unit Profit (All Orders)')
ax1.legend()

# Bar: avg unit profit by discount band
bands  = discount_impact.index.astype(str)
values = discount_impact['Avg_Unit_Profit']
colors = ['seagreen' if v >= 0 else 'crimson' for v in values]
ax2.bar(bands, values, color=colors)
ax2.axhline(0, color='black', linewidth=0.8)
ax2.set_xlabel('Discount Band')
ax2.set_ylabel('Avg Unit Profit ($)')
ax2.set_title('Average Unit Profit by Discount Band')

plt.tight_layout()
plt.show()


**Chart 5 — Gross Profit Distribution per Order Line** *(KPI 1)*  
*Why it exists: Shows the shape of profitability — specifically how many orders are loss-making and by how much.*

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

ax.hist(sales['Gross Profit'], bins=80, color='steelblue', edgecolor='none', alpha=0.8)
ax.axvline(0, color='crimson', linewidth=1.5, linestyle='--', label='Break-even ($0)')
ax.axvline(sales['Gross Profit'].median(), color='goldenrod', linewidth=1.5, linestyle='--',
           label=f'Median (${sales["Gross Profit"].median():,.0f})')

ax.set_xlabel('Gross Profit per Line Item ($)')
ax.set_ylabel('Number of Orders')
ax.set_title('Gross Profit Distribution')
ax.legend()
plt.tight_layout()
plt.show()


## 8. Key Observations

- **Negative margin exposure is minimal but real.** Only 1 product (0.39%) has a weighted negative margin, creating a 0.006% profitability drag — not a crisis, but worth investigating whether it's a pricing error or an intentional loss leader.

- **Revenue and profit move together, but margin fluctuates.** Monthly trend shows no long-term divergence between revenue and profit, indicating healthy alignment — growth appears broadly sustainable.

- **Office Supplies dominates revenue but not margin.** Furniture delivers a comparatively stronger margin despite lower absolute revenue. Office Supplies shows the weakest margin-to-revenue ratio.

- **Top-selling products are not uniformly profitable.** Several high-revenue products carry thin margins — high volume is masking a weak per-unit return. This is where pricing review should start.

- **Discounting past ~6% erodes unit profit noticeably.** The scatter and band analysis shows unit profit declining sharply in the 6–10% discount range. A discount guardrail policy is worth considering.
- **Overall Decline.** The data is showing clear sharp decline in overall revenue after 2016. Which must be thoroughly investigated. 
- **Not all categories are performing well.** The data is showing that Office Supplies is approx 47 times Furniture and approx 5 times Technology. So focusing on the Office Supplies will give best ROI. 